# 02 — Baseline evaluation: Mistral-7B-Instruct before fine-tuning

In [5]:
import json
import wandb
from bert_score import score
from mlx_lm import load, generate

/Users/harthikmallichetty/Desktop/code-sentinel/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Matplotlib is building the font cache; this may take a moment.
/Users/harthikmallichetty/Desktop/code-sentinel/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def load_examples(file_path, n):
    """
    load_examples(file_path, n)

    The function takes in a file and number of samples to be loaded.

    input: string, integer
    output: data loaded from preprocessed JSON files
    """
    data = []
    counter = 0
    with open(file_path, "r") as file:
        for line in file:
            data.append(json.loads(line))
            counter += 1
            if counter == n:
                break
    return data

def format_prompt(row):
    """
    format_prompt(row)

    This function takes in a row from the JSONL file and outputs a prompt with diff and lang.

    input: JSONL object
    output: string
    """
    return f'''[INST] Review the following code change and identify issues:
Language: {row["lang"]}
Diff: {row["hunk"]}
Provide specific, actionable feedback.[/INST]'''

In [10]:
model, tokenizer = load("mistralai/Mistral-7B-Instruct-v0.3")

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 28728.11it/s]


In [9]:
predictions = [] 
references = []
examples = load_examples("/Users/harthikmallichetty/Desktop/code-sentinel-data-source/ref-test.jsonl", 500)

for line in examples:
    references.append(line["comment"]) 
    predictions.append(generate(model, tokenizer, format_prompt(line)))

In [14]:
P, R, F1 = score(predictions, references, lang="en", model_type="distilbert-base-uncased")
print(f"Average F1 Score for the Base Model: {F1.mean().item()}")


Average F1 Score for the Base Model: 0.7148823738098145


In [17]:
wandb.init(project="code-sentinel", name="baseline-mistral-7b")
wandb.log({"baseline_bertscore_f1": 0.7148823738098145})
wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/harthikmallichetty/.netrc.
wandb: Currently logged in as: harthikmallichetty (harthikmallichetty-the-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


baseline_bertscore_f1,▁
baseline_bertscore_f1,0.71488
